In [0]:
# =========================================================
# CELL 1: CONFIGURATION
# Notebook: 07_new_relic_pipeline_health
# =========================================================

import json
import hashlib
from datetime import datetime, timezone

import requests

from pyspark.sql import functions as F
from delta.tables import DeltaTable


# ---------------------------------------------------------
# TIMEZONE
# ---------------------------------------------------------

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)


# ---------------------------------------------------------
# PROJECT CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
VOLUME = "volume1"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

PIPELINE_NAME = "cold_chain_databricks_pipeline"
TEAM_NAME = "cold-chain-capstone"
ENVIRONMENT = "training"


# ---------------------------------------------------------
# TABLE NAMES
# ---------------------------------------------------------

AUDIT_TABLE = (
    f"{NAMESPACE}.pipeline_run_audit"
)

BRONZE_SENSOR_TABLE = (
    f"{NAMESPACE}.bronze_sensor_events"
)

SILVER_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_events"
)

QUARANTINE_SENSOR_TABLE = (
    f"{NAMESPACE}.quarantine_sensor_events"
)

ENRICHED_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_enriched"
)

TEMPERATURE_WINDOWS_TABLE = (
    f"{NAMESPACE}.shipment_temperature_windows"
)

SHIPMENT_HEALTH_TABLE = (
    f"{NAMESPACE}.shipment_health"
)

DELIVERY_PERFORMANCE_TABLE = (
    f"{NAMESPACE}.delivery_performance"
)

TELEMETRY_LOG_TABLE = (
    f"{NAMESPACE}.new_relic_telemetry_log"
)


# ---------------------------------------------------------
# SECRET CONFIGURATION
# ---------------------------------------------------------

NEW_RELIC_SECRET_SCOPE = "newrelic"
NEW_RELIC_SECRET_KEY = "license-key"


# ---------------------------------------------------------
# WIDGET HELPERS
# ---------------------------------------------------------

def create_text_widget_if_missing(
    widget_name,
    default_value
):
    try:
        dbutils.widgets.get(widget_name)
    except Exception:
        dbutils.widgets.text(
            widget_name,
            default_value
        )


def create_dropdown_widget_if_missing(
    widget_name,
    default_value,
    allowed_values
):
    try:
        dbutils.widgets.get(widget_name)
    except Exception:
        dbutils.widgets.dropdown(
            widget_name,
            default_value,
            allowed_values
        )


create_text_widget_if_missing(
    "run_id",
    ""
)

create_dropdown_widget_if_missing(
    "new_relic_region",
    "US",
    ["US", "EU", "JP"]
)

create_dropdown_widget_if_missing(
    "publish_enabled",
    "false",
    ["true", "false"]
)


# ---------------------------------------------------------
# READ WIDGETS
# ---------------------------------------------------------

REQUESTED_RUN_ID = (
    dbutils.widgets
    .get("run_id")
    .strip()
)

NEW_RELIC_REGION = (
    dbutils.widgets
    .get("new_relic_region")
    .strip()
    .upper()
)

PUBLISH_ENABLED = (
    dbutils.widgets
    .get("publish_enabled")
    .strip()
    .lower()
    == "true"
)


assert NEW_RELIC_REGION in {
    "US",
    "EU",
    "JP"
}, (
    "new_relic_region must be US, EU, or JP"
)


print("==============================================")
print("NEW RELIC PIPELINE HEALTH")
print("==============================================")
print("Namespace          :", NAMESPACE)
print("Pipeline           :", PIPELINE_NAME)
print("Team               :", TEAM_NAME)
print("Environment        :", ENVIRONMENT)
print("Requested run ID   :", REQUESTED_RUN_ID or "latest")
print("New Relic region   :", NEW_RELIC_REGION)
print("Publishing enabled :", PUBLISH_ENABLED)

In [0]:
# =========================================================
# CELL 2: SELECT PIPELINE RUN
# =========================================================

print("==============================================")
print("AUDIT TABLE VALIDATION")
print("==============================================")


assert spark.catalog.tableExists(
    AUDIT_TABLE
), (
    f"Required audit table is missing: {AUDIT_TABLE}"
)


audit_df = spark.table(
    AUDIT_TABLE
)


# ---------------------------------------------------------
# REQUIRED CORE AUDIT COLUMNS
# ---------------------------------------------------------

required_core_audit_columns = {
    "run_id",
    "status",
    "started_at",
    "completed_at",
    "input_rows",
    "accepted_rows",
    "rejected_rows",
    "rejected_percent",
    "error_message"
}


missing_core_columns = (
    required_core_audit_columns
    - set(audit_df.columns)
)


assert not missing_core_columns, (
    "Audit table is missing required columns: "
    f"{sorted(missing_core_columns)}"
)


# ---------------------------------------------------------
# OPTIONAL COLUMNS
# These are strongly recommended but the notebook remains
# backward-compatible if they have not yet been added.
# ---------------------------------------------------------

recommended_audit_columns = {
    "orchestrator",
    "airflow_dag_id",
    "airflow_run_id",
    "databricks_job_id",
    "databricks_run_id",

    "bronze_status",
    "silver_status",
    "gold_status",

    "failed_stage",
    "failed_task",
    "error_type",
    "retry_count",

    "temperature_exception_count",
    "gold_temperature_window_rows",
    "shipment_health_rows",
    "delivery_performance_rows"
}


missing_recommended_columns = (
    recommended_audit_columns
    - set(audit_df.columns)
)


if missing_recommended_columns:
    print(
        "WARNING: Recommended audit columns are missing:"
    )

    for column_name in sorted(
        missing_recommended_columns
    ):
        print(" -", column_name)


# ---------------------------------------------------------
# SELECT REQUESTED OR LATEST TERMINAL RUN
# ---------------------------------------------------------

if REQUESTED_RUN_ID:

    selected_audit_df = (
        audit_df
        .filter(
            F.col("run_id") == REQUESTED_RUN_ID
        )
        .orderBy(
            F.col("completed_at")
            .desc_nulls_last()
        )
        .limit(1)
    )

else:

    selected_audit_df = (
        audit_df
        .filter(
            F.col("status").isin(
                "SUCCESS",
                "FAILED",
                "PARTIAL_SUCCESS",
                "CANCELLED",
                "TIMED_OUT"
            )
        )
        .orderBy(
            F.coalesce(
                F.col("completed_at"),
                F.col("started_at")
            ).desc()
        )
        .limit(1)
    )


selected_rows = selected_audit_df.collect()


assert len(selected_rows) == 1, (
    "No matching terminal pipeline run was found. "
    "Provide a valid run_id or ensure the audit table "
    "contains a completed SUCCESS or FAILED run."
)


audit_run = selected_rows[0]

RUN_ID = str(
    audit_run["run_id"]
)

PIPELINE_STATUS = str(
    audit_run["status"] or "UNKNOWN"
).upper()

started_at = audit_run["started_at"]
completed_at = audit_run["completed_at"]


assert started_at is not None, (
    "The selected audit record has no started_at value."
)


# A failed run should normally have completed_at.
# The fallback prevents telemetry from being lost.
if completed_at is None:

    completed_at = datetime.now(
        timezone.utc
    ).replace(
        tzinfo=None
    )

    print(
        "WARNING: completed_at is missing. "
        "Telemetry execution time is being used as "
        "the fallback completion time."
    )


print("==============================================")
print("SELECTED PIPELINE RUN")
print("==============================================")
print("Run ID       :", RUN_ID)
print("Status       :", PIPELINE_STATUS)
print("Started at   :", started_at)
print("Completed at :", completed_at)

display(selected_audit_df)

In [0]:
# =========================================================
# CELL 3: HELPERS AND JOB CONTEXT
# =========================================================

def safe_int(
    value,
    default_value=0
):
    if value is None:
        return default_value

    try:
        return int(value)
    except Exception:
        return default_value


def safe_float(
    value,
    default_value=None
):
    if value is None:
        return default_value

    try:
        return float(value)
    except Exception:
        return default_value


def audit_value(
    column_name,
    default_value=None
):
    """
    Safely read an audit-table value.

    This keeps the notebook compatible while new audit
    columns are being introduced.
    """

    if column_name not in audit_df.columns:
        return default_value

    value = audit_run[column_name]

    if value is None:
        return default_value

    return value


def normalize_status(
    value,
    default_value="UNKNOWN"
):
    if value is None:
        return default_value

    normalized = str(value).strip().upper()

    return normalized or default_value


def iso_utc(value):
    """
    Convert Python datetime to UTC ISO text.
    """

    if value is None:
        return None

    if value.tzinfo is None:
        value = value.replace(
            tzinfo=timezone.utc
        )

    return value.astimezone(
        timezone.utc
    ).isoformat()


def remove_none_values(event):
    """
    New Relic does not store null-valued attributes.
    """

    return {
        key: value
        for key, value in event.items()
        if value is not None
    }


def get_context_tag(tag_name):
    try:

        context = (
            dbutils.notebook
            .entry_point
            .getDbutils()
            .notebook()
            .getContext()
        )

        value = (
            context
            .tags()
            .apply(tag_name)
        )

        if value is None:
            return None

        value = str(value).strip()

        return value if value else None

    except Exception:
        return None


# ---------------------------------------------------------
# READ ORCHESTRATION DETAILS
# ---------------------------------------------------------

ORCHESTRATOR = str(
    audit_value(
        "orchestrator",
        "databricks"
    )
)

AIRFLOW_DAG_ID = audit_value(
    "airflow_dag_id"
)

AIRFLOW_RUN_ID = audit_value(
    "airflow_run_id"
)


audit_job_id = audit_value(
    "databricks_job_id"
)

audit_databricks_run_id = audit_value(
    "databricks_run_id"
)


DATABRICKS_JOB_ID = (
    get_context_tag("jobId")
    or get_context_tag("job_id")
    or (
        str(audit_job_id)
        if audit_job_id is not None
        else None
    )
)


DATABRICKS_RUN_ID = (
    get_context_tag("jobRunId")
    or get_context_tag("runId")
    or get_context_tag("job_run_id")
    or (
        str(audit_databricks_run_id)
        if audit_databricks_run_id is not None
        else None
    )
)


DATABRICKS_TASK_KEY = (
    get_context_tag("taskKey")
    or get_context_tag("task_key")
)


# ---------------------------------------------------------
# READ LAYER AND FAILURE STATUS
# ---------------------------------------------------------

BRONZE_STATUS = normalize_status(
    audit_value(
        "bronze_status"
    )
)

SILVER_STATUS = normalize_status(
    audit_value(
        "silver_status"
    )
)

GOLD_STATUS = normalize_status(
    audit_value(
        "gold_status"
    )
)


FAILED_STAGE = str(
    audit_value(
        "failed_stage",
        ""
    )
)

FAILED_TASK = str(
    audit_value(
        "failed_task",
        ""
    )
)

ERROR_TYPE = str(
    audit_value(
        "error_type",
        ""
    )
)

ERROR_MESSAGE = str(
    audit_value(
        "error_message",
        ""
    )
)[:1000]

RETRY_COUNT = safe_int(
    audit_value(
        "retry_count",
        0
    )
)


# ---------------------------------------------------------
# FALLBACK FAILURE VALUES
# These do not invent a cause. They only indicate that
# detailed diagnostic fields were not populated.
# ---------------------------------------------------------

if PIPELINE_STATUS == "FAILED":

    if not FAILED_STAGE:
        FAILED_STAGE = "UNKNOWN"

    if not FAILED_TASK:
        FAILED_TASK = (
            DATABRICKS_TASK_KEY
            or "UNKNOWN"
        )

    if not ERROR_TYPE:
        ERROR_TYPE = "UNCLASSIFIED_ERROR"

    if not ERROR_MESSAGE:
        ERROR_MESSAGE = (
            "Pipeline failed, but no error message "
            "was recorded in the pipeline audit table."
        )


print("==============================================")
print("ORCHESTRATION CONTEXT")
print("==============================================")
print("Orchestrator      :", ORCHESTRATOR)
print("Airflow DAG ID    :", AIRFLOW_DAG_ID)
print("Airflow Run ID    :", AIRFLOW_RUN_ID)
print("Databricks Job ID :", DATABRICKS_JOB_ID)
print("Databricks Run ID :", DATABRICKS_RUN_ID)
print("Databricks Task   :", DATABRICKS_TASK_KEY)

print()
print("Layer Status")
print("Bronze :", BRONZE_STATUS)
print("Silver :", SILVER_STATUS)
print("Gold   :", GOLD_STATUS)

print()
print("Failure Diagnostics")
print("Failed Stage :", FAILED_STAGE or "None")
print("Failed Task  :", FAILED_TASK or "None")
print("Error Type   :", ERROR_TYPE or "None")
print("Retry Count  :", RETRY_COUNT)

In [0]:
# =========================================================
# CELL 4: CALCULATE ACTUAL PIPELINE METRICS
# =========================================================

def table_or_empty(
    table_name,
    logical_name
):
    """
    Read a table if it exists.

    Failed pipeline runs may not have produced all
    downstream tables.
    """

    if spark.catalog.tableExists(
        table_name
    ):
        print(
            f"{logical_name}: FOUND"
        )

        return spark.table(
            table_name
        )

    print(
        f"{logical_name}: NOT AVAILABLE"
    )

    return None


def scope_to_selected_run(
    dataframe,
    logical_name
):
    if dataframe is None:
        return None

    columns = set(
        dataframe.columns
    )

    if "run_id" in columns:

        print(
            f"{logical_name}: scoped using run_id"
        )

        return dataframe.filter(
            F.col("run_id") == RUN_ID
        )

    if "_processing_time" in columns:

        print(
            f"{logical_name}: scoped using "
            "_processing_time"
        )

        return dataframe.filter(
            (
                F.col("_processing_time")
                >= F.lit(started_at)
            )
            &
            (
                F.col("_processing_time")
                <= F.lit(completed_at)
            )
        )

    print(
        f"WARNING: {logical_name} has no run_id or "
        "_processing_time. Full-table values may not "
        "represent only run {RUN_ID}."
    )

    return dataframe


# ---------------------------------------------------------
# READ PIPELINE TABLES
# ---------------------------------------------------------

bronze_sensor_df = table_or_empty(
    BRONZE_SENSOR_TABLE,
    "Bronze sensors"
)

silver_sensor_df = table_or_empty(
    SILVER_SENSOR_TABLE,
    "Silver sensors"
)

quarantine_sensor_df = table_or_empty(
    QUARANTINE_SENSOR_TABLE,
    "Quarantine sensors"
)

enriched_sensor_df = table_or_empty(
    ENRICHED_SENSOR_TABLE,
    "Enriched sensors"
)

temperature_windows_df = table_or_empty(
    TEMPERATURE_WINDOWS_TABLE,
    "Temperature windows"
)

shipment_health_df = table_or_empty(
    SHIPMENT_HEALTH_TABLE,
    "Shipment health"
)

delivery_performance_df = table_or_empty(
    DELIVERY_PERFORMANCE_TABLE,
    "Delivery performance"
)


# ---------------------------------------------------------
# SCOPE TABLES TO THE SELECTED RUN
# ---------------------------------------------------------

run_bronze_sensor_df = scope_to_selected_run(
    bronze_sensor_df,
    "Bronze sensors"
)

run_silver_sensor_df = scope_to_selected_run(
    silver_sensor_df,
    "Silver sensors"
)

run_quarantine_sensor_df = scope_to_selected_run(
    quarantine_sensor_df,
    "Quarantine sensors"
)

run_enriched_sensor_df = scope_to_selected_run(
    enriched_sensor_df,
    "Enriched sensors"
)

run_temperature_windows_df = scope_to_selected_run(
    temperature_windows_df,
    "Temperature windows"
)

run_shipment_health_df = scope_to_selected_run(
    shipment_health_df,
    "Shipment health"
)

run_delivery_performance_df = scope_to_selected_run(
    delivery_performance_df,
    "Delivery performance"
)


# ---------------------------------------------------------
# RUN-LEVEL COUNTS FROM AUDIT TABLE
# ---------------------------------------------------------

input_rows = safe_int(
    audit_value(
        "input_rows",
        0
    )
)

accepted_rows = safe_int(
    audit_value(
        "accepted_rows",
        0
    )
)

rejected_rows = safe_int(
    audit_value(
        "rejected_rows",
        0
    )
)


audit_rejected_percent = safe_float(
    audit_value(
        "rejected_percent"
    )
)


if audit_rejected_percent is not None:

    rejected_percent = (
        audit_rejected_percent
    )

elif input_rows > 0:

    rejected_percent = round(
        rejected_rows
        / input_rows
        * 100.0,
        2
    )

else:

    rejected_percent = None


temperature_exception_count = safe_int(
    audit_value(
        "temperature_exception_count",
        0
    )
)

gold_temperature_window_rows = safe_int(
    audit_value(
        "gold_temperature_window_rows",
        0
    )
)

shipment_health_rows = safe_int(
    audit_value(
        "shipment_health_rows",
        0
    )
)

delivery_performance_rows = safe_int(
    audit_value(
        "delivery_performance_rows",
        0
    )
)


gold_output_rows = (
    gold_temperature_window_rows
    + shipment_health_rows
    + delivery_performance_rows
)


# Accepted rows are the validated Silver output.
output_rows = accepted_rows

no_data_condition = (
    input_rows == 0
)


# ---------------------------------------------------------
# DUPLICATE COUNT
# ---------------------------------------------------------

duplicate_rows = 0


if (
    run_bronze_sensor_df is not None
    and "event_id"
    in run_bronze_sensor_df.columns
):

    duplicate_result = (
        run_bronze_sensor_df
        .filter(
            F.col("event_id").isNotNull()
        )
        .groupBy("event_id")
        .count()
        .filter(
            F.col("count") > 1
        )
        .select(
            F.sum(
                F.col("count") - F.lit(1)
            ).alias(
                "duplicate_rows"
            )
        )
        .first()
    )

    if duplicate_result is not None:
        duplicate_rows = safe_int(
            duplicate_result[
                "duplicate_rows"
            ]
        )


duplicate_percent = (
    round(
        duplicate_rows
        / input_rows
        * 100.0,
        2
    )
    if input_rows > 0
    else None
)


# ---------------------------------------------------------
# BUSINESS HEALTH COUNTS
# ---------------------------------------------------------

missing_sensor_count = 0


if (
    run_shipment_health_df is not None
    and "missing_sensor_status"
    in run_shipment_health_df.columns
):

    missing_sensor_count = (
        run_shipment_health_df
        .filter(
            F.upper(
                F.col("missing_sensor_status")
            ).isin(
                "MISSING",
                "STALE"
            )
        )
        .count()
    )


open_overdue_count = 0


if (
    run_delivery_performance_df is not None
    and "open_overdue_status"
    in run_delivery_performance_df.columns
):

    open_overdue_count = (
        run_delivery_performance_df
        .filter(
            F.upper(
                F.col("open_overdue_status")
            )
            == "OPEN_OVERDUE"
        )
        .count()
    )


# ---------------------------------------------------------
# DURATION
# ---------------------------------------------------------

duration_seconds = max(
    0.0,
    float(
        (
            completed_at
            - started_at
        ).total_seconds()
    )
)


# ---------------------------------------------------------
# LATEST EVENT TIME
# ---------------------------------------------------------

latest_event_timestamp = None


if (
    run_enriched_sensor_df is not None
    and "event_time"
    in run_enriched_sensor_df.columns
):

    latest_event_row = (
        run_enriched_sensor_df
        .select(
            F.expr(
                "try_cast(event_time AS TIMESTAMP)"
            ).alias(
                "parsed_event_time"
            )
        )
        .filter(
            F.col(
                "parsed_event_time"
            ).isNotNull()
        )
        .agg(
            F.max(
                "parsed_event_time"
            ).alias(
                "latest_event_timestamp"
            )
        )
        .first()
    )

    if latest_event_row is not None:

        latest_event_timestamp = (
            latest_event_row[
                "latest_event_timestamp"
            ]
        )


# ---------------------------------------------------------
# LATEST PROCESSING TIME
# ---------------------------------------------------------

latest_processing_timestamp = None


if (
    run_bronze_sensor_df is not None
    and "_processing_time"
    in run_bronze_sensor_df.columns
):

    latest_processing_row = (
        run_bronze_sensor_df
        .agg(
            F.max(
                "_processing_time"
            ).alias(
                "latest_processing_timestamp"
            )
        )
        .first()
    )

    if latest_processing_row is not None:

        latest_processing_timestamp = (
            latest_processing_row[
                "latest_processing_timestamp"
            ]
        )


# ---------------------------------------------------------
# FRESHNESS
# ---------------------------------------------------------

freshness_seconds = None


if latest_event_timestamp is not None:

    freshness_seconds = max(
        0.0,
        float(
            (
                completed_at
                - latest_event_timestamp
            ).total_seconds()
        )
    )


# ---------------------------------------------------------
# SOURCE-TO-PROCESSING DELAY
# ---------------------------------------------------------

source_to_processing_delay_seconds = None


if (
    run_bronze_sensor_df is not None
    and {
        "event_time",
        "_processing_time"
    }.issubset(
        set(
            run_bronze_sensor_df.columns
        )
    )
):

    delay_row = (
        run_bronze_sensor_df
        .withColumn(
            "parsed_event_time",
            F.expr(
                "try_cast(event_time AS TIMESTAMP)"
            )
        )
        .filter(
            F.col(
                "parsed_event_time"
            ).isNotNull()
            &
            F.col(
                "_processing_time"
            ).isNotNull()
        )
        .withColumn(
            "delay_seconds",
            F.unix_timestamp(
                "_processing_time"
            )
            -
            F.unix_timestamp(
                "parsed_event_time"
            )
        )
        .filter(
            F.col(
                "delay_seconds"
            ) >= 0
        )
        .agg(
            F.max(
                "delay_seconds"
            ).alias(
                "maximum_delay_seconds"
            )
        )
        .first()
    )

    if (
        delay_row is not None
        and delay_row[
            "maximum_delay_seconds"
        ] is not None
    ):

        source_to_processing_delay_seconds = (
            safe_float(
                delay_row[
                    "maximum_delay_seconds"
                ]
            )
        )


# ---------------------------------------------------------
# REJECTION REASONS
# ---------------------------------------------------------

rejection_reason_counts = {}


if (
    run_quarantine_sensor_df is not None
    and "rejection_reason"
    in run_quarantine_sensor_df.columns
):

    rejection_reason_rows = (
        run_quarantine_sensor_df
        .filter(
            F.col(
                "rejection_reason"
            ).isNotNull()
        )
        .withColumn(
            "normalized_reason",
            F.lower(
                F.trim(
                    F.col(
                        "rejection_reason"
                    )
                )
            )
        )
        .groupBy(
            "normalized_reason"
        )
        .count()
        .orderBy(
            "normalized_reason"
        )
        .collect()
    )

    rejection_reason_counts = {
        str(
            row[
                "normalized_reason"
            ]
        ): safe_int(
            row["count"]
        )
        for row
        in rejection_reason_rows
    }


mandatory_field_reasons = {
    "missing_event_id",
    "missing_shipment_id",
    "missing_event_time"
}

invalid_data_type_reasons = {
    "nonnumeric_temperature",
    "invalid_temperature",
    "invalid_event_time"
}

reference_validation_reasons = {
    "unknown_shipment",
    "unknown_product"
}

schema_validation_reasons = {
    "schema_mismatch",
    "missing_required_column",
    "unexpected_column"
}


mandatory_field_failures = sum(
    count_value
    for reason, count_value
    in rejection_reason_counts.items()
    if reason in mandatory_field_reasons
)

invalid_data_type_failures = sum(
    count_value
    for reason, count_value
    in rejection_reason_counts.items()
    if reason in invalid_data_type_reasons
)

reference_validation_failures = sum(
    count_value
    for reason, count_value
    in rejection_reason_counts.items()
    if reason in reference_validation_reasons
)

schema_validation_failures = sum(
    count_value
    for reason, count_value
    in rejection_reason_counts.items()
    if reason in schema_validation_reasons
)


# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------

print("==============================================")
print("ACTUAL PIPELINE HEALTH METRICS")
print("==============================================")
print("Run ID                    :", RUN_ID)
print("Pipeline status           :", PIPELINE_STATUS)
print("Bronze status             :", BRONZE_STATUS)
print("Silver status             :", SILVER_STATUS)
print("Gold status               :", GOLD_STATUS)
print("Duration seconds          :", duration_seconds)
print("Input rows                :", input_rows)
print("Accepted rows             :", accepted_rows)
print("Rejected rows             :", rejected_rows)
print("Duplicate rows            :", duplicate_rows)
print("Rejected percentage       :", rejected_percent)
print("Duplicate percentage      :", duplicate_percent)
print("Gold output rows          :", gold_output_rows)
print("Freshness seconds         :", freshness_seconds)
print("Temperature exceptions    :", temperature_exception_count)
print("Missing/stale sensors     :", missing_sensor_count)
print("Open overdue shipments    :", open_overdue_count)

In [0]:
# =========================================================
# CELL 5: BUILD NEW RELIC EVENTS
# =========================================================

# ---------------------------------------------------------
# EVENT 1: PIPELINE EXECUTION HEALTH
# ---------------------------------------------------------

pipeline_event = {

    "eventType": "ColdChainPipelineRun",

    # Identification
    "team": TEAM_NAME,
    "pipeline": PIPELINE_NAME,
    "environment": ENVIRONMENT,
    "runId": RUN_ID,

    # Orchestration
    "orchestrator": ORCHESTRATOR,
    "airflowDagId": AIRFLOW_DAG_ID,
    "airflowRunId": AIRFLOW_RUN_ID,
    "databricksJobId": DATABRICKS_JOB_ID,
    "databricksRunId": DATABRICKS_RUN_ID,
    "databricksTaskKey": DATABRICKS_TASK_KEY,

    # Overall pipeline health
    "status": PIPELINE_STATUS,

    # Layer health
    "bronzeStatus": BRONZE_STATUS,
    "silverStatus": SILVER_STATUS,
    "goldStatus": GOLD_STATUS,

    # Execution timing
    "startedAt": iso_utc(
        started_at
    ),
    "completedAt": iso_utc(
        completed_at
    ),
    "durationSeconds": duration_seconds,

    # Data movement
    "inputRows": input_rows,
    "outputRows": output_rows,
    "acceptedRows": accepted_rows,
    "rejectedRows": rejected_rows,
    "duplicateRows": duplicate_rows,

    # Data quality
    "rejectedPercent": rejected_percent,
    "duplicatePercent": duplicate_percent,

    # Freshness
    "latestEventTimestamp": iso_utc(
        latest_event_timestamp
    ),
    "latestProcessingTimestamp": iso_utc(
        latest_processing_timestamp
    ),
    "freshnessSeconds": freshness_seconds,
    "sourceToProcessingDelaySeconds": (
        source_to_processing_delay_seconds
    ),
    "noDataCondition": no_data_condition,

    # Pipeline output counts
    "bronzeSensorRows": input_rows,
    "silverAcceptedRows": accepted_rows,
    "silverRejectedRows": rejected_rows,
    "quarantineRows": rejected_rows,

    "goldTemperatureWindowRows": (
        gold_temperature_window_rows
    ),
    "shipmentHealthRows": (
        shipment_health_rows
    ),
    "deliveryPerformanceRows": (
        delivery_performance_rows
    ),
    "goldOutputRows": (
        gold_output_rows
    ),

    # Business calculations
    "temperatureExceptionCount": (
        temperature_exception_count
    ),
    "missingSensorCount": (
        missing_sensor_count
    ),
    "openOverdueCount": (
        open_overdue_count
    ),

    # Gold readiness
    "businessKpiGenerationStatus": (
        GOLD_STATUS
    ),
    "goldRefreshTime": iso_utc(
        completed_at
    ),

    # Failure diagnostics
    "failedStage": FAILED_STAGE,
    "failedTask": FAILED_TASK,
    "errorType": ERROR_TYPE,
    "errorMessage": ERROR_MESSAGE,
    "retryCount": RETRY_COUNT
}


# ---------------------------------------------------------
# EVENT 2: DATA QUALITY
# ---------------------------------------------------------

data_quality_event = {

    "eventType": "ColdChainDataQuality",

    "team": TEAM_NAME,
    "pipeline": PIPELINE_NAME,
    "environment": ENVIRONMENT,
    "runId": RUN_ID,

    "databricksJobId": (
        DATABRICKS_JOB_ID
    ),

    "databricksRunId": (
        DATABRICKS_RUN_ID
    ),

    "pipelineStatus": (
        PIPELINE_STATUS
    ),

    "inputRows": input_rows,
    "acceptedRows": accepted_rows,
    "rejectedRows": rejected_rows,
    "duplicateRows": duplicate_rows,

    "rejectedPercent": (
        rejected_percent
    ),

    "duplicatePercent": (
        duplicate_percent
    ),

    "schemaValidationFailures": (
        schema_validation_failures
    ),

    "mandatoryFieldFailures": (
        mandatory_field_failures
    ),

    "referenceValidationFailures": (
        reference_validation_failures
    ),

    "invalidDataTypeFailures": (
        invalid_data_type_failures
    ),

    "evaluatedAt": iso_utc(
        completed_at
    )
}


# ---------------------------------------------------------
# EVENT 3: BUSINESS HEALTH
# ---------------------------------------------------------

business_health_event = {

    "eventType": "ColdChainBusinessHealth",

    "team": TEAM_NAME,
    "pipeline": PIPELINE_NAME,
    "environment": ENVIRONMENT,
    "runId": RUN_ID,

    "databricksJobId": (
        DATABRICKS_JOB_ID
    ),

    "databricksRunId": (
        DATABRICKS_RUN_ID
    ),

    "pipelineStatus": (
        PIPELINE_STATUS
    ),

    "goldStatus": (
        GOLD_STATUS
    ),

    "temperatureExceptionCount": (
        temperature_exception_count
    ),

    "missingSensorCount": (
        missing_sensor_count
    ),

    "openOverdueCount": (
        open_overdue_count
    ),

    "shipmentHealthRows": (
        shipment_health_rows
    ),

    "deliveryPerformanceRows": (
        delivery_performance_rows
    ),

    "evaluatedAt": iso_utc(
        completed_at
    )
}


# ---------------------------------------------------------
# EVENT 4: REJECTION REASON EVENTS
# ---------------------------------------------------------

rejection_events = []


for reason, count_value in (
    rejection_reason_counts.items()
):

    rejection_events.append({

        "eventType":
            "ColdChainRejectionReason",

        "team":
            TEAM_NAME,

        "pipeline":
            PIPELINE_NAME,

        "environment":
            ENVIRONMENT,

        "runId":
            RUN_ID,

        "databricksJobId":
            DATABRICKS_JOB_ID,

        "databricksRunId":
            DATABRICKS_RUN_ID,

        "dataset":
            "sensor_events",

        "rejectionReason":
            str(reason),

        "rejectedRows":
            safe_int(
                count_value
            ),

        "inputRows":
            input_rows,

        "evaluatedAt":
            iso_utc(
                completed_at
            )
    })


# ---------------------------------------------------------
# COMBINE AND CLEAN
# ---------------------------------------------------------

new_relic_events = [
    pipeline_event,
    data_quality_event,
    business_health_event
] + rejection_events


new_relic_events = [
    remove_none_values(
        event
    )
    for event
    in new_relic_events
]


assert len(
    new_relic_events
) >= 3, (
    "At least three New Relic events must be created."
)


payload_json = json.dumps(
    new_relic_events,
    sort_keys=True,
    separators=(",", ":"),
    default=str
)


print("==============================================")
print("NEW RELIC TELEMETRY PAYLOAD")
print("==============================================")
print("Events prepared:", len(new_relic_events))
print()


print(
    json.dumps(
        new_relic_events,
        indent=2,
        default=str
    )
)

In [0]:
# =========================================================
# CELL 6: LOG TABLE AND NEW RELIC CONFIGURATION
# =========================================================

print("==============================================")
print("TELEMETRY LOG INITIALIZATION")
print("==============================================")


# ---------------------------------------------------------
# CREATE TELEMETRY LOG TABLE
# ---------------------------------------------------------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {TELEMETRY_LOG_TABLE}
    (
        telemetry_id STRING,
        run_id STRING,
        databricks_job_id STRING,
        databricks_run_id STRING,
        payload_hash STRING,
        event_count BIGINT,
        pipeline_status STRING,
        publish_status STRING,
        http_status BIGINT,
        published_at TIMESTAMP,
        error_message STRING
    )
    USING DELTA
    """
)


# Add pipeline_status if this is an older table.
existing_log_columns = {
    field.name
    for field
    in spark.table(
        TELEMETRY_LOG_TABLE
    ).schema.fields
}


if "pipeline_status" not in existing_log_columns:

    spark.sql(
        f"""
        ALTER TABLE {TELEMETRY_LOG_TABLE}
        ADD COLUMNS (
            pipeline_status STRING
        )
        """
    )


assert spark.catalog.tableExists(
    TELEMETRY_LOG_TABLE
)


# ---------------------------------------------------------
# PAYLOAD HASH
# ---------------------------------------------------------

payload_hash = hashlib.sha256(
    payload_json.encode(
        "utf-8"
    )
).hexdigest()


telemetry_id = (
    f"{RUN_ID}:{payload_hash}"
)


print("Telemetry ID :", telemetry_id)
print("Payload Hash :", payload_hash)


# ---------------------------------------------------------
# DUPLICATE PUBLICATION CHECK
# ---------------------------------------------------------

existing_success_count = (
    spark.table(
        TELEMETRY_LOG_TABLE
    )
    .filter(
        (
            F.col("telemetry_id")
            == telemetry_id
        )
        &
        (
            F.col("publish_status")
            == "SUCCESS"
        )
    )
    .count()
)


ALREADY_PUBLISHED = (
    existing_success_count > 0
)


print(
    "Already Published:",
    ALREADY_PUBLISHED
)


# ---------------------------------------------------------
# NEW RELIC REGIONAL ENDPOINTS
# ---------------------------------------------------------

region_endpoints = {

    "US": (
        "https://insights-collector.newrelic.com"
        "/v1/accounts/events"
    ),

    "EU": (
        "https://insights-collector.eu01.nr-data.net"
        "/v1/accounts/events"
    ),

    "JP": (
        "https://insights-collector.jp.nr-data.net"
        "/v1/accounts/events"
    )
}


NEW_RELIC_EVENT_URL = (
    region_endpoints[
        NEW_RELIC_REGION
    ]
)


print("Region   :", NEW_RELIC_REGION)
print("Endpoint :", NEW_RELIC_EVENT_URL)


# ---------------------------------------------------------
# RETRIEVE THE KEY SECURELY
# ---------------------------------------------------------

NEW_RELIC_LICENSE_KEY = None


if PUBLISH_ENABLED:

    try:

        NEW_RELIC_LICENSE_KEY = (
            dbutils.secrets.get(
                scope=NEW_RELIC_SECRET_SCOPE,
                key=NEW_RELIC_SECRET_KEY
            )
        )

    except Exception as exc:

        raise RuntimeError(
            "Unable to retrieve New Relic license key "
            "from the Databricks secret scope. "
            f"Scope={NEW_RELIC_SECRET_SCOPE}, "
            f"Key={NEW_RELIC_SECRET_KEY}. "
            "Create the secret before enabling publication."
        ) from exc


    assert (
        NEW_RELIC_LICENSE_KEY is not None
        and len(
            NEW_RELIC_LICENSE_KEY.strip()
        ) > 0
    ), (
        "New Relic license key is blank."
    )


    print(
        "New Relic key retrieved securely."
    )

else:

    print(
        "Publishing is disabled, so the New Relic "
        "secret was not retrieved."
    )


print("==============================================")
print("CELL 6 COMPLETED")
print("==============================================")

In [0]:
# =========================================================
# CELL 7: PUBLISH AND LOG RESULT
# =========================================================

print("==============================================")
print("NEW RELIC PUBLICATION")
print("==============================================")


publication_status = "SKIPPED"
http_status = None
publication_error = None


# ---------------------------------------------------------
# CASE 1: PUBLISHING DISABLED
# ---------------------------------------------------------

if not PUBLISH_ENABLED:

    publication_status = "DISABLED"

    print(
        "Publishing is disabled."
    )

    print(
        "The payload was created but not sent."
    )


# ---------------------------------------------------------
# CASE 2: IDENTICAL PAYLOAD ALREADY PUBLISHED
# ---------------------------------------------------------

elif ALREADY_PUBLISHED:

    publication_status = (
        "ALREADY_PUBLISHED"
    )

    print(
        "This identical payload was previously "
        "published successfully."
    )

    print(
        "Duplicate publication was skipped."
    )


# ---------------------------------------------------------
# CASE 3: SEND TO NEW RELIC
# ---------------------------------------------------------

else:

    try:

        print(
            "Publishing telemetry to New Relic..."
        )

        response = requests.post(

            NEW_RELIC_EVENT_URL,

            headers={
                "Api-Key":
                    NEW_RELIC_LICENSE_KEY,

                "Content-Type":
                    "application/json"
            },

            data=payload_json,

            timeout=30
        )


        http_status = safe_int(
            response.status_code
        )


        print(
            "HTTP Status:",
            http_status
        )


        if not (
            200
            <= response.status_code
            < 300
        ):

            raise RuntimeError(
                "New Relic Event API returned "
                f"HTTP {response.status_code}. "
                f"Response: {response.text[:500]}"
            )


        publication_status = "SUCCESS"


        print(
            "Telemetry published successfully."
        )


    except Exception as exc:

        publication_status = "FAILED"

        publication_error = str(
            exc
        )[:2000]


        print(
            "Telemetry publication failed."
        )

        print(
            "Error:",
            publication_error
        )


# ---------------------------------------------------------
# PUBLICATION TIMESTAMP
# ---------------------------------------------------------

published_at = (
    datetime.now(
        timezone.utc
    )
    .replace(
        tzinfo=None
    )
)


# ---------------------------------------------------------
# SAVE PUBLICATION RESULT
# ---------------------------------------------------------

if publication_status != (
    "ALREADY_PUBLISHED"
):

    telemetry_log_rows = [(
        telemetry_id,
        RUN_ID,
        DATABRICKS_JOB_ID,
        DATABRICKS_RUN_ID,
        payload_hash,
        int(
            len(
                new_relic_events
            )
        ),
        PIPELINE_STATUS,
        publication_status,
        http_status,
        published_at,
        publication_error
    )]


    telemetry_log_df = (
        spark.createDataFrame(
            telemetry_log_rows,
            """
            telemetry_id STRING,
            run_id STRING,
            databricks_job_id STRING,
            databricks_run_id STRING,
            payload_hash STRING,
            event_count BIGINT,
            pipeline_status STRING,
            publish_status STRING,
            http_status BIGINT,
            published_at TIMESTAMP,
            error_message STRING
            """
        )
    )


    telemetry_log_delta = (
        DeltaTable.forName(
            spark,
            TELEMETRY_LOG_TABLE
        )
    )


    (
        telemetry_log_delta
        .alias("target")
        .merge(
            telemetry_log_df.alias(
                "source"
            ),
            """
            target.telemetry_id =
            source.telemetry_id
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


    print(
        "Publication result saved."
    )


else:

    print(
        "Existing successful publication "
        "record was retained."
    )


# ---------------------------------------------------------
# DISPLAY CURRENT RUN LOG
# ---------------------------------------------------------

display(
    spark.table(
        TELEMETRY_LOG_TABLE
    )
    .filter(
        F.col(
            "run_id"
        ) == RUN_ID
    )
    .orderBy(
        F.col(
            "published_at"
        ).desc()
    )
)


print("==============================================")
print("PIPELINE STATUS    :", PIPELINE_STATUS)
print("PUBLICATION STATUS :", publication_status)
print("HTTP STATUS        :", http_status)
print("==============================================")

In [0]:
# =========================================================
# CELL 8: FINAL VALIDATION
# =========================================================

print("==============================================")
print("NEW RELIC PIPELINE HEALTH CERTIFICATE")
print("==============================================")


# ---------------------------------------------------------
# IDENTITY
# ---------------------------------------------------------

print("Pipeline Name           :", PIPELINE_NAME)
print("Run ID                  :", RUN_ID)
print("Environment             :", ENVIRONMENT)
print("Orchestrator            :", ORCHESTRATOR)


# ---------------------------------------------------------
# PIPELINE STATUS
# ---------------------------------------------------------

print()
print("PIPELINE HEALTH")
print("----------------------------------------------")
print("Overall Status          :", PIPELINE_STATUS)
print("Bronze Status           :", BRONZE_STATUS)
print("Silver Status           :", SILVER_STATUS)
print("Gold Status             :", GOLD_STATUS)


# ---------------------------------------------------------
# JOB INFORMATION
# ---------------------------------------------------------

print()
print("JOB INFORMATION")
print("----------------------------------------------")
print("Airflow DAG ID          :", AIRFLOW_DAG_ID)
print("Airflow Run ID          :", AIRFLOW_RUN_ID)
print("Databricks Job ID       :", DATABRICKS_JOB_ID)
print("Databricks Run ID       :", DATABRICKS_RUN_ID)
print("Databricks Task Key     :", DATABRICKS_TASK_KEY)


# ---------------------------------------------------------
# EXECUTION
# ---------------------------------------------------------

print()
print("EXECUTION")
print("----------------------------------------------")
print("Started At              :", started_at)
print("Completed At            :", completed_at)
print("Duration Seconds        :", duration_seconds)
print("Retry Count             :", RETRY_COUNT)


# ---------------------------------------------------------
# DATA MOVEMENT
# ---------------------------------------------------------

print()
print("DATA MOVEMENT")
print("----------------------------------------------")
print("Input Rows              :", input_rows)
print("Accepted Rows           :", accepted_rows)
print("Rejected Rows           :", rejected_rows)
print("Duplicate Rows          :", duplicate_rows)
print("Output Rows             :", output_rows)
print("Gold Output Rows        :", gold_output_rows)


# ---------------------------------------------------------
# DATA QUALITY
# ---------------------------------------------------------

print()
print("DATA QUALITY")
print("----------------------------------------------")
print("Rejected Percent        :", rejected_percent)
print("Duplicate Percent       :", duplicate_percent)
print("Schema Failures         :", schema_validation_failures)
print("Mandatory Failures      :", mandatory_field_failures)
print("Reference Failures      :", reference_validation_failures)
print("Data Type Failures      :", invalid_data_type_failures)


# ---------------------------------------------------------
# FRESHNESS
# ---------------------------------------------------------

print()
print("FRESHNESS")
print("----------------------------------------------")
print("Latest Event Timestamp  :", latest_event_timestamp)
print("Latest Processing Time  :", latest_processing_timestamp)
print("Freshness Seconds       :", freshness_seconds)
print("Source Delay Seconds    :", source_to_processing_delay_seconds)


# ---------------------------------------------------------
# BUSINESS HEALTH
# ---------------------------------------------------------

print()
print("BUSINESS HEALTH")
print("----------------------------------------------")
print("Temperature Exceptions  :", temperature_exception_count)
print("Missing/Stale Sensors   :", missing_sensor_count)
print("Open Overdue Shipments  :", open_overdue_count)


# ---------------------------------------------------------
# FAILURE INFORMATION
# ---------------------------------------------------------

print()
print("FAILURE INFORMATION")
print("----------------------------------------------")
print("Failed Stage            :", FAILED_STAGE or "None")
print("Failed Task             :", FAILED_TASK or "None")
print("Error Type              :", ERROR_TYPE or "None")
print("Error Message           :", ERROR_MESSAGE or "None")


# ---------------------------------------------------------
# PUBLICATION
# ---------------------------------------------------------

print()
print("NEW RELIC PUBLICATION")
print("----------------------------------------------")
print("Events Prepared         :", len(new_relic_events))
print("Publication Status      :", publication_status)
print("HTTP Status             :", http_status)


# ---------------------------------------------------------
# BASIC VALIDATION
# ---------------------------------------------------------

assert input_rows >= 0
assert accepted_rows >= 0
assert rejected_rows >= 0
assert duplicate_rows >= 0
assert output_rows >= 0
assert gold_output_rows >= 0
assert duration_seconds >= 0


if rejected_percent is not None:

    assert (
        0
        <= rejected_percent
        <= 100
    ), (
        "Rejected percentage must be "
        "between 0 and 100."
    )


if duplicate_percent is not None:

    assert (
        0
        <= duplicate_percent
        <= 100
    ), (
        "Duplicate percentage must be "
        "between 0 and 100."
    )


if freshness_seconds is not None:

    assert freshness_seconds >= 0


# ---------------------------------------------------------
# COUNT RECONCILIATION
# ---------------------------------------------------------
# Duplicates are treated as an informational metric here
# because duplicate handling may differ between pipelines.

assert (
    accepted_rows + rejected_rows
) <= input_rows, (
    "Row reconciliation failed.\n"
    f"Input Rows    : {input_rows}\n"
    f"Accepted Rows : {accepted_rows}\n"
    f"Rejected Rows : {rejected_rows}"
)


# ---------------------------------------------------------
# FAILURE FIELD VALIDATION
# ---------------------------------------------------------

if PIPELINE_STATUS == "FAILED":

    assert FAILED_STAGE, (
        "Failed pipeline run must contain failedStage."
    )

    assert ERROR_TYPE, (
        "Failed pipeline run must contain errorType."
    )

    assert ERROR_MESSAGE, (
        "Failed pipeline run must contain errorMessage."
    )


# ---------------------------------------------------------
# SUCCESSFUL GOLD RUN VALIDATION
# ---------------------------------------------------------

if (
    PIPELINE_STATUS == "SUCCESS"
    and GOLD_STATUS == "SUCCESS"
):

    assert (
        gold_temperature_window_rows > 0
    ), (
        "Gold-status is SUCCESS, but the temperature "
        "window row count is zero."
    )

    assert (
        shipment_health_rows > 0
    ), (
        "Gold-status is SUCCESS, but shipment health "
        "row count is zero."
    )

    assert (
        delivery_performance_rows > 0
    ), (
        "Gold-status is SUCCESS, but delivery "
        "performance row count is zero."
    )


# ---------------------------------------------------------
# PUBLICATION VALIDATION
# ---------------------------------------------------------

acceptable_publication_statuses = {
    "SUCCESS",
    "ALREADY_PUBLISHED",
    "DISABLED"
}


assert publication_status in (
    acceptable_publication_statuses
), (
    "New Relic publication was not successful. "
    f"Publication status: {publication_status}. "
    f"Error: {publication_error}"
)


print()
print("==============================================")
print("TELEMETRY VALIDATION PASSED")
print("==============================================")
print("Run ID              :", RUN_ID)
print("Pipeline Status     :", PIPELINE_STATUS)
print("Publication Status  :", publication_status)
print("Telemetry Events    :", len(new_relic_events))
print("Freshness Seconds   :", freshness_seconds)
print("==============================================")

In [0]:
%sql

DESCRIBE TABLE training_lab_classic.cold_chain.pipeline_run_audit;

In [0]:
%sql

SELECT
    run_id,
    status,
    bronze_status,
    silver_status,
    gold_status,
    failed_stage,
    failed_task,
    error_type,
    error_message,
    retry_count
FROM training_lab_classic.cold_chain.pipeline_run_audit
WHERE run_id = 'manual_001';

In [0]:
%sql

UPDATE training_lab_classic.cold_chain.pipeline_run_audit
SET
    bronze_status = 'SUCCESS',
    silver_status = 'SUCCESS',
    gold_status = 'SUCCESS',
    failed_stage = NULL,
    failed_task = NULL,
    error_type = NULL,
    error_message = NULL,
    retry_count = 0
WHERE run_id = 'manual_001';

In [0]:
%sql

SELECT
    run_id,
    status,
    bronze_status,
    silver_status,
    gold_status,
    failed_stage,
    failed_task,
    error_type,
    error_message,
    retry_count
FROM training_lab_classic.cold_chain.pipeline_run_audit
WHERE run_id = 'manual_001';